# Generare video

Il codice del capitolo [«Generare video»](https://book.paithon.it/main/GenerazioneVideo/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Generare video

[Leggi la pagina](https://book.paithon.it/main/GenerazioneVideo/overview.html)


### Un video è un tensore


In [ ]:
# Un video come tensore F x H x W x 3, contato in valori.
secondi, fps = 5, 24
F, H, W = secondi * fps, 720, 1280

per_fotogramma = H * W * 3
per_video = F * per_fotogramma
gib = per_video * 4 / 2**30          # un valore in float32 occupa 4 byte

print(f"fotogrammi F: {F}")
print(f"valori in un fotogramma: {per_fotogramma}")
print(f"valori in {secondi} s di video: {per_video}")
print(f"memoria in float32: {gib:.2f} GiB")
print(f"valori in un minuto: {60 * fps * per_fotogramma}")
print(f"pixel di un clip 1280x768 da 128 fotogrammi: {128 * 1280 * 768}")

### Perché non basta generare immagini una per una


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
# W pixel per riga, F fotogrammi, SIGMA deviazione standard del picco,
# PASSO pixel di spostamento per fotogramma
W, F, SIGMA, PASSO = 64, 16, 3.0, 1.0
asse = np.arange(W)

def fotogramma(pos):
    d = np.abs(asse - pos)
    d = np.minimum(d, W - d)                    # distanza sul cerchio
    return np.exp(-0.5 * (d / SIGMA) ** 2)

def video(posizioni):
    return np.stack([fotogramma(p) for p in posizioni])

n_video = 2000
# indipendente: ogni fotogramma ha la sua posizione a caso
indipendenti = np.array([video(rng.uniform(0, W, F)) for _ in range(n_video)])
# un secondo campione della stessa sorgente: misura il rumore della stima
altri = np.array([video(rng.uniform(0, W, F)) for _ in range(n_video)])
# coerente: parte da un punto a caso e avanza di PASSO a ogni fotogramma
coerenti = np.array([video((rng.uniform(0, W) + PASSO * np.arange(F)) % W)
                     for _ in range(n_video)])
# fermo: la stessa posizione a caso in tutti i fotogrammi
fermi = np.array([video(np.full(F, rng.uniform(0, W)))
                  for _ in range(n_video)])

def sfarfallio(v):
    # differenza media assoluta fra fotogrammi adiacenti, pixel per pixel
    return np.abs(np.diff(v, axis=1)).mean()

m = 7   # un fotogramma qualsiasi: l'ottavo, perche' Python conta da zero
for nome, v in [("altri indip.", altri), ("coerente", coerenti), ("fermo", fermi)]:
    scarto = np.abs(v[:, m].mean(axis=0) - indipendenti[:, m].mean(axis=0)).max()
    print(f"{nome:12} scarto massimo della media al fotogramma {m + 1}: {scarto:.3f}")
# il caso da solo: 200 confronti fra due campioni della stessa sorgente,
# ciascuno di n_video fotogrammi con il picco in un punto a caso
def media_a_caso():
    return fotogramma(rng.uniform(0, W, (n_video, 1))).mean(axis=0)
massimi = [np.abs(media_a_caso() - media_a_caso()).max() for _ in range(200)]
print(f"stessa sorgente, 200 confronti: mediana {np.median(massimi):.3f}, "
      f"massimo {np.max(massimi):.3f}")
for nome, v in [("indipendente", indipendenti), ("coerente", coerenti), ("fermo", fermi)]:
    print(f"sfarfallio {nome:12}: {sfarfallio(v):.4f}")

## Comprimere il tempo: l’autoencoder video

[Leggi la pagina](https://book.paithon.it/main/GenerazioneVideo/comprimere-il-tempo.html)


### Quanti numeri restano: dal clip ai token


In [ ]:
import math

def conta(F, H, W, ct, cs, C, ph=2, pw=2, primo_a_parte=True):
    """Valori nei pixel, nel latente e token dopo la patch (pt = 1)."""
    if primo_a_parte:
        Fl = 1 + (F - 1) // ct          # primo fotogramma da solo
    else:
        Fl = math.ceil(F / ct)          # gruppi tutti uguali
    pixel = F * H * W * 3
    latente = Fl * (H // cs) * (W // cs) * C
    S = (H // (cs * ph)) * (W // (cs * pw))   # token per fotogramma latente
    return pixel, latente, Fl, S, Fl * S

# Wan: 81 fotogrammi a 1280x720, VAE 4x8x8 con 16 canali, patch 1x2x2
pixel, lat, Fl, S, N = conta(81, 720, 1280, 4, 8, 16)
immagine = 1 * 720 * 1280 * 3
print("immagine 1280x720:", immagine, "valori")
print("clip di 81 fotogrammi:", pixel, "valori")
print("rapporto clip/immagine:", round(pixel / immagine, 1))
print("latente:", Fl, "fotogrammi latenti,", lat, "valori")
print("compressione:", round(pixel / lat, 1), "(senza il +1 sarebbe 48)")
print("token per fotogramma latente S =", S, " token totali N =", N)

# HunyuanVideo: 129 fotogrammi 720x1280, stessi fattori
p, l, Fl, S, N = conta(129, 720, 1280, 4, 8, 16)
print("Hunyuan: F' =", Fl, " S =", S, " N =", N)

# Movie Gen: 256 fotogrammi 768x768, 8x8x8, 16 canali, primo non a parte
p, l, Fl, S, N = conta(256, 768, 768, 8, 8, 16, primo_a_parte=False)
print("Movie Gen: F' =", Fl, " S =", S, " N =", N)

### La convoluzione causale e il primo fotogramma


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def conv_causale(x, k, s):
    """Convoluzione temporale causale con kernel k e passo s.
    x ha forma (F, ...): si aggiungono k-1 fotogrammi nulli solo prima."""
    pad = np.zeros((k - 1,) + x.shape[1:])
    xp = np.concatenate([pad, x])
    w = np.arange(1, k + 1, dtype=float)       # pesi fissi, solo per la prova
    n_out = (len(xp) - k) // s + 1
    return np.stack([sum(w[j] * xp[i * s + j] for j in range(k))
                     for i in range(n_out)])

ct = 4
F = 9
x = rng.normal(size=(F, 3))                    # 9 fotogrammi, 3 numeri ciascuno
z = conv_causale(x, k=ct, s=ct)
print("fotogrammi:", F, "-> latenti:", len(z),
      " (1 + 8/4 =", 1 + (F - 1) // ct, ")")

# da quali fotogrammi dipende ogni latente? Si altera un fotogramma alla volta
# (nella stampa i fotogrammi e i latenti sono numerati da 1).
for i in range(len(z)):
    dip = []
    for f in range(F):
        z_alt = conv_causale(x + np.eye(F)[f][:, None], ct, ct)
        if not np.allclose(z_alt[i], z[i]):
            dip.append(f)
    print("latente", i + 1, "dipende dai fotogrammi", [f + 1 for f in dip])

# prova diretta: alterando i fotogrammi dal sesto in poi
# i primi due latenti non cambiano
y = x.copy()
y[5:] += 10.0
zy = conv_causale(y, ct, ct)
print("primi due latenti invariati:", np.array_equal(z[:2], zy[:2]),
      "| terzo cambiato:", not np.array_equal(z[2], zy[2]))

# un'immagine e' un video di un fotogramma
print("F = 1 ->", len(conv_causale(x[:1], ct, ct)), "latente")

# lunghezze che non sono 1 piu' un multiplo di 4: l'ultimo fotogramma coperto
for F2 in (9, 8, 10):
    n = len(conv_causale(rng.normal(size=(F2, 1)), ct, ct))
    print("F =", F2, "-> latenti:", n, " ultimo fotogramma coperto:",
          (n - 1) * ct + 1, "su", F2)

### Video lunghi: la cache delle feature


In [ ]:
import numpy as np

rng = np.random.default_rng(1)
w = [0.5, 0.3, 0.2]                       # kernel temporale di lunghezza 3

def passo(finestra):
    """Un valore di uscita da tre fotogrammi, in un ordine fissato."""
    return w[0] * finestra[0] + w[1] * finestra[1] + w[2] * finestra[2]

def tutto_insieme(x):
    xp = [0.0, 0.0] + list(x)             # due fotogrammi nulli solo prima
    return [passo(xp[i:i + 3]) for i in range(len(x))]

def a_blocchi(x, dimensioni):
    cache = [0.0, 0.0]                    # primo blocco: due fotogrammi nulli
    uscita, inizio = [], 0
    for d in dimensioni:                  # un blocco, un fotogramma latente
        blocco = list(x[inizio:inizio + d])
        xp = cache + blocco
        uscita += [passo(xp[i:i + 3]) for i in range(len(blocco))]
        cache = xp[-2:]                   # si conservano gli ultimi due
        inizio += d
    return uscita, cache

F = 9
x = rng.normal(size=F)
dimensioni = [1] + [4] * ((F - 1) // 4)   # 1 + 8/4 = 3 blocchi: 1, 4, 4
print("blocchi:", dimensioni)
a = tutto_insieme(x)
b, cache = a_blocchi(x, dimensioni)
print("uscite identiche:", a == b)
print("fotogrammi tenuti dalla cache:", len(cache), "(contro", F, "del clip)")

### Quanto comprimere: il compromesso fra token e dettagli


In [ ]:
# (nome, ct, cs, canali, patch p_t, p_h, p_w)
vae = [
    ("HunyuanVideo", 4,  8,  16, 1, 2, 2),
    ("Wan 2.1",     4,  8,  16, 1, 2, 2),
    ("Movie Gen",   8,  8,  16, 1, 2, 2),
    ("LTX-Video",   8, 32, 128, 1, 1, 1),
]
print("modello       fattore  pixel per token")
for nome, ct, cs, C, pt, ph, pw in vae:
    fattore = ct * cs * cs * 3 / C            # valori pixel / valori latente
    pixel_per_token = ct * cs * cs * pt * ph * pw
    print(f"{nome:13s} {fattore:6.0f}  {pixel_per_token:8d}")

# lo stesso clip, 768x512 con 121 fotogrammi: LTX contro il VAE di Wan
F, H, W = 121, 512, 768
Fl_ltx = 1 + (F - 1) // 8
S_ltx = (H // 32) * (W // 32)
Fl_wan = 1 + (F - 1) // 4
S_wan = (H // 16) * (W // 16)                  # VAE 8x8 e patch 2x2
N_ltx, N_wan = Fl_ltx * S_ltx, Fl_wan * S_wan
print("LTX-Video:", Fl_ltx, "latenti x", S_ltx, "=", N_ltx, "token")
print("VAE di Wan:", Fl_wan, "latenti x", S_wan, "=", N_wan, "token")
print("rapporto fra i token:", round(N_wan / N_ltx, 2))
# costo relativo dell'attenzione piena: cresce come N^2
print("rapporto fra i costi di attenzione:", round((N_wan / N_ltx) ** 2))

## L’attenzione nello spazio e nel tempo

[Leggi la pagina](https://book.paithon.it/main/GenerazioneVideo/spazio-e-tempo.html)


### Innestare il tempo in un modello di immagini: l’attenzione fattorizzata


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
Fl, S, d = 4, 9, 16
z = rng.normal(size=(Fl, S, d))           # uscita di uno strato spaziale
Wq, Wk, Wv = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))

def con_tempo(z, Wo):
    """Attenzione temporale: i posti spaziali fanno da batch, il tempo da sequenza."""
    q, k, v = ((z @ W).transpose(1, 0, 2) for W in (Wq, Wk, Wv))   # (S, F', d)
    p = np.exp(q @ k.transpose(0, 2, 1) / np.sqrt(d))
    p /= p.sum(-1, keepdims=True)
    return z + (p @ v @ Wo).transpose(1, 0, 2)                     # residuo

def scarto(a, b):
    return np.abs(a - b).max()

zero = np.zeros((d, d))                    # proiezione di uscita inizializzata a zero
print("proiezione a zero, scarto dall'uscita spaziale:", scarto(con_tempo(z, zero), z))

Wo = rng.normal(size=(d, d)) / np.sqrt(d)  # dopo l'addestramento sarebbe diversa
zt = con_tempo(z, Wo)
for alfa in (1.0, 0.5):
    fuso = alfa * z + (1 - alfa) * zt      # fusione di Video LDM
    print(f"alfa = {alfa}: scarto dal modello di immagini = {scarto(fuso, z):.3f}")
print(f"massimo modulo dell'uscita spaziale: {np.abs(z).max():.3f}")

In [ ]:
import numpy as np

Fl, S = 4, 9                       # fotogrammi latenti e token per fotogramma
N = Fl * S
n = np.arange(N)
fotogramma, posto = n // S, n % S   # il token n sta nel fotogramma n // S

spaziale = fotogramma[:, None] == fotogramma[None, :]   # stesso fotogramma
temporale = posto[:, None] == posto[None, :]            # stesso posto
piena = np.ones((N, N), dtype=bool)

print("coppie, attenzione piena:", piena.sum())
print("coppie, spaziale + temporale:", spaziale.sum() + temporale.sum(),
      f"({spaziale.sum()} + {temporale.sum()})")
print(f"rapporto: {piena.sum() / (spaziale.sum() + temporale.sum()):.2f}",
      f"(F'S/(F'+S) = {Fl * S / (Fl + S):.2f})")

# q: fotogramma 2, angolo (posto 0). x: fotogramma 1, centro (posto 4)
q, x = 2 * S + 0, 1 * S + 4
print("q guarda x in uno strato, piena:", piena[q, x])
print("q guarda x in uno strato, spaziale:", spaziale[q, x])
print("q guarda x in uno strato, temporale:", temporale[q, x])
due = (temporale.astype(int) @ spaziale.astype(int)) > 0   # spaziale, poi temporale
print("dopo spaziale e temporale, q riceve da x:", due[q, x])
print("coppie raggiunte dopo i due strati:", due.sum(), "su", N * N)

### Tutti i token in una sola sequenza: l’attenzione 3D piena


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
gruppi = (16, 56, 56)                  # canali per fotogramma, riga, colonna
dk = sum(gruppi)                       # 128 canali, la testa di HunyuanVideo

def ruota(v, pos):
    """RoPE a tre componenti: ogni gruppo di canali ruota con la sua coordinata."""
    out, a = [], 0
    for g, p in zip(gruppi, pos):
        ang = p * 10000.0 ** (-np.arange(g // 2) / (g // 2))
        x1, x2 = v[a:a + g:2], v[a + 1:a + g:2]
        r = np.empty(g)
        r[0::2] = x1 * np.cos(ang) - x2 * np.sin(ang)
        r[1::2] = x1 * np.sin(ang) + x2 * np.cos(ang)
        out.append(r)
        a += g
    return np.concatenate(out)

q, k = rng.normal(size=dk), rng.normal(size=dk)
def punteggio(pn, pm):
    return ruota(q, pn) @ ruota(k, pm)

a = punteggio((2, 5, 7), (1, 3, 9))        # differenze (-1, -2, +2)
b = punteggio((10, 0, 20), (9, -2, 22))    # stesse differenze, altre posizioni
c = punteggio((2, 5, 7), (1, 3, 10))       # una differenza cambiata: colonna +3
print(f"stesse differenze, posizioni diverse: {a:.6f} {b:.6f}")
print(f"differenza in colonna cambiata:       {c:.6f}")
print("coppie ruotate per gruppo:", [g // 2 for g in gruppi])

### Il costo dell’attenzione piena rispetto al resto dello strato


In [ ]:
# Wan 14B (README di Wan 2.1): larghezza d e strato feed-forward dff
d, dff = 5120, 13824
print(f"2d + dff = {2 * d + dff:,}")

def rapporto(N, d, dff):
    attenzione = 2 * N**2 * d                  # Q K^T e prodotto con V
    lineare = 4 * N * d**2 + 2 * N * d * dff   # proiezioni e feed-forward
    return attenzione / lineare

for N in (75_600, 1_000_000):
    r = rapporto(N, d, dff)
    print(f"N = {N:>9,}: attenzione/resto = {r:5.2f}, "
          f"quota dello strato = {r / (1 + r):.1%}")

# con dff = 4d il rapporto e' N/(6d), come nel DiT
print("dff = 4d, N = 75600:", round(rapporto(75_600, d, 4 * d), 2),
      "contro N/(6d) =", round(75_600 / (6 * d), 2))

# la cross-attention di Wan verso 512 token di testo: proiezioni di query e
# uscita sul video, di chiavi e valori sul testo, e i punteggi N x 512
N, Nt = 75_600, 512
croce = 2 * N * d**2 + 2 * Nt * d**2 + 2 * N * Nt * d
r = 2 * N**2 * d / (4 * N * d**2 + 2 * N * d * dff + croce)
print(f"con la cross-attention: attenzione/resto = {r:.2f}, "
      f"quota dello strato = {r / (1 + r):.1%}")

# fattorizzata contro piena sul clip di Wan: 21 fotogrammi latenti, 3600 token
Fl, S = 21, 3600
N = Fl * S
piena, fattorizzata = N**2, Fl * S**2 + S * Fl**2
print(f"coppie: piena {piena:,}, fattorizzata {fattorizzata:,}")
print(f"rapporto {piena / fattorizzata:.1f}, F'S/(F'+S) = {N / (Fl + S):.1f}")

# blocco fattorizzato: un'attenzione spaziale e una temporale (due serie
# di proiezioni, 8 N d^2) e un feed-forward
blocco_piena = 2 * N**2 * d + 4 * N * d**2 + 2 * N * d * dff
punteggi = 2 * N * (S + Fl) * d
blocco_fatt = punteggi + 8 * N * d**2 + 2 * N * d * dff
print(f"fattorizzata: punteggi = {punteggi / blocco_fatt:.1%} del blocco")
print(f"operazioni del blocco, piena / fattorizzata = "
      f"{blocco_piena / blocco_fatt:.2f}")

### Come entra il testo: cross-attention o flusso unico


In [ ]:
# coppie di attenzione: il clip di Wan (75.600 token) e 512 token di testo
N, Nt = 75_600, 512
print(f"cross-attention: N x Nt = {N * Nt:,}, "
      f"{N * Nt / N**2:.2%} delle coppie della self-attention")
print(f"flusso unico: (N + Nt)^2 = {(N + Nt)**2:,}, "
      f"{(N + Nt)**2 / N**2 - 1:.2%} di coppie in piu'")

# modulazione dal passo t: uno strato lineare d -> 6d in ogni blocco,
# oppure uno solo condiviso piu' un bias di 6d numeri per blocco
for nome, L, d in (("DiT-XL/2", 28, 1152), ("Wan 1.3B", 30, 1536),
                   ("Wan 14B", 40, 5120)):
    per_blocco = L * 6 * d * d
    condiviso = 6 * d * d + L * 6 * d
    print(f"{nome:9}: un MLP per blocco {per_blocco / 1e9:.2f} miliardi "
          f"di pesi, condiviso {condiviso / 1e6:.1f} milioni")

## Insegnare il movimento: dati, addestramento, misura

[Leggi la pagina](https://book.paithon.it/main/GenerazioneVideo/addestrare-e-misurare.html)


### L’obiettivo e quanto rumore mettere


In [ ]:
import numpy as np

rng = np.random.default_rng(3)
freq = np.arange(4)                  # coseni a bassa frequenza: segnale liscio

def campo(m, n):
    """n immagini m x m, somma di coseni a bassa frequenza, varianza 1."""
    u = (np.arange(m) + 0.5) / m
    c = np.cos(np.pi * np.outer(freq, u))
    a = rng.normal(size=(n, 4, 4))
    x = np.einsum("nij,iy,jx->nyx", a, c, c)
    return x / x.std()

def errore(x, z, sigma):
    """Errore quadratico dello stimatore a blocchi 8 x 8."""
    n, m, _ = x.shape
    b = m // 8
    media = (x + sigma * z).reshape(n, 8, b, 8, b).mean(axis=(2, 4))
    stima = np.repeat(np.repeat(media, b, axis=1), b, axis=2)
    return float(np.mean((stima - x) ** 2))

dati = {}
for m in (16, 32, 64, 128):
    x = campo(m, 24)
    dati[m] = (x, rng.normal(size=x.shape))

print("lato   errore a sigma = 1")
for m, (x, z) in dati.items():
    print(f"{m:4d}   {errore(x, z, 1.0):.4f}")

bersaglio = errore(*dati[16], 1.0)
print("\nlato   sigma che pareggia   t = 0.5 spostato a")
for m in (32, 64, 128):
    lo, hi = 0.5, 20.0
    for _ in range(30):
        mid = (lo + hi) / 2
        if errore(*dati[m], mid) < bersaglio:
            lo = mid
        else:
            hi = mid
    print(f"{m:4d}   {lo:12.2f}   {lo / (1 + lo):18.3f}")

### Che cosa si dà da vedere al modello: dati, didascalie, tappe


In [ ]:
F, c_t, c_s = 81, 4, 8                # fotogrammi, compressione del VAE
patch_h = patch_w = 2                 # patch 1 x 2 x 2

def token(H, W):
    fl = 1 + (F - 1) // c_t           # fotogrammi latenti
    return fl * (H // c_s // patch_h) * (W // c_s // patch_w)

piccolo, grande = token(480, 832), token(720, 1280)
print("token a 832x480  :", piccolo)
print("token a 1280x720 :", grande)
print(f"costo lineare    : x{grande / piccolo:.2f}")
print(f"costo quadratico : x{(grande / piccolo) ** 2:.2f}")

tenuti, iniziali = 152, 577           # milioni di clip: LVD-F e LVD (Stable Video Diffusion)
print(f"\nclip tenuti dai filtri: {tenuti / iniziali:.3f}")
print(f"clip scartati         : {1 - tenuti / iniziali:.3f}")

### Misurare un video: FVD, VBench e il video fermo


In [ ]:
import numpy as np

rng = np.random.default_rng(11)
F, n = 16, 160                        # fotogrammi, pixel della scena 1D
x = np.arange(n)

def video(v, rumore=0.02):
    """Una macchia che si sposta di v pixel a fotogramma, con rumore del sensore."""
    c0 = rng.uniform(10, n - 10 - v * F)
    centri = c0 + v * np.arange(F)
    f = np.exp(-0.5 * ((x[None, :] - centri[:, None]) / 3.0) ** 2)
    return f + rumore * rng.normal(size=f.shape), centri

def coerenza(f):
    """Similarita' coseno media fra fotogrammi vicini (sui pixel, non su DINO)."""
    a, b = f[:-1], f[1:]
    cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    return cos.mean()

def dinamico(centri, soglia=0.5):
    """Dinamico: la macchia si sposta in media piu' di 'soglia' pixel a fotogramma."""
    return np.abs(np.diff(centri)).mean() > soglia

print("velocita'  coerenza  dinamica")
for v in (0.0, 1.0, 2.0, 3.0):
    prove = [video(v) for _ in range(200)]
    c = np.mean([coerenza(f) for f, _ in prove])
    d = np.mean([dinamico(ce) for _, ce in prove])
    print(f"{v:8.1f}  {c:8.3f}  {d:8.2f}")

# fotogrammi indipendenti: ognuno e' una scena plausibile, fra loro nessun legame
indip = [np.stack([video(0.0)[0][0] for _ in range(F)]) for _ in range(200)]
print(f"\nfotogrammi indipendenti: coerenza {np.mean([coerenza(f) for f in indip]):.3f}")

In [ ]:
import numpy as np

rng = np.random.default_rng(5)
F, n = 16, 160
x = np.arange(n)

def clip():
    """Macchia in moto, da 1 a 3 pixel a fotogramma, in un verso o nell'altro."""
    v = rng.uniform(1, 3) * rng.choice([-1, 1])
    c0 = rng.uniform(10, n - 10 - abs(v) * F) + (abs(v) * F if v < 0 else 0)
    centri = c0 + v * np.arange(F)
    f = np.exp(-0.5 * ((x[None, :] - centri[:, None]) / 3.0) ** 2)
    return f + 0.02 * rng.normal(size=f.shape)

def mescola(f):
    """Stessi fotogrammi, ordine temporale casuale."""
    return f[rng.permutation(F)]

def solo_contenuto(f):
    """Che cosa c'e' nel clip, senza l'ordine: media dei fotogrammi, in 8 fasce."""
    return f.mean(0).reshape(8, -1).mean(1)

def con_moto(f):
    """Contenuto, piu' la differenza media assoluta fra fotogrammi vicini."""
    d = np.abs(np.diff(f, axis=0)).mean(0).reshape(8, -1).mean(1)
    return np.concatenate([solo_contenuto(f), d])

def frechet(a, b):
    """Distanza di Frechet fra due gaussiane stimate sui due insiemi di feature."""
    ma, mb = a.mean(0), b.mean(0)
    sa, sb = np.cov(a.T), np.cov(b.T)
    autov = np.linalg.eigvals(sa @ sb).real.clip(min=0)
    return float(((ma - mb) ** 2).sum() + np.trace(sa) + np.trace(sb)
                 - 2 * np.sqrt(autov).sum())

print("        solo contenuto         contenuto+moto")
print("prova  vero/vero  mescolato   vero/vero  mescolato  rapporto")
for prova in range(1, 6):             # cinque estrazioni indipendenti
    vere1 = [clip() for _ in range(600)]
    vere2 = [clip() for _ in range(600)]
    mescolati = [mescola(f) for f in vere2]
    d = []
    for feat in (solo_contenuto, con_moto):
        A = np.array([feat(f) for f in vere1])
        B = np.array([feat(f) for f in vere2])
        C = np.array([feat(f) for f in mescolati])
        d += [frechet(A, B), frechet(A, C)]
    print(f"{prova:5d}  {d[0]:9.5f}  {d[1]:9.5f}   {d[2]:9.5f}  {d[3]:9.5f}"
          f"  {d[3] / d[2]:8.0f}")

## Generare mentre il video scorre: tempo reale e mondi

[Leggi la pagina](https://book.paithon.it/main/GenerazioneVideo/generare-mentre-scorre.html)


### Imparare dal proprio passato: exposure bias e tre regimi di addestramento


In [ ]:
import numpy as np

# Un "video" è un'onda. Il modello continua con la ricorrenza
# x[i+1] = 2 cos(w) x[i] - x[i-1], con una frequenza stimata il 2% troppo alta.
n = 300
omega = 2 * np.pi / 60
c = 2 * np.cos(1.02 * omega)                       # il modello
c_vero = 2 * np.cos(omega)                         # la dinamica vera
x = np.sin(omega * np.arange(n))                   # il video vero

# addestramento: ogni passo parte dal passato vero
errore_vero = np.abs(c * x[1:-1] - x[:-2] - x[2:]).mean()

# uso: ogni passo parte dai fotogrammi che il modello ha già prodotto
y = list(x[:2])
for i in range(2, n):
    y.append(c * y[-1] - y[-2])
y = np.array(y)
# lo sbaglio di ogni passo, quando il passato è quello del modello
sbagli = np.abs((c - c_vero) * y[1:-1])
distanza = np.abs(y - x)

print(f"errore di un passo, passato vero:    {errore_vero:.5f}")
print(f"errore di un passo, passato proprio: {sbagli.mean():.5f}")
for i in (10, 50, 100, 200, 299):
    print(f"distanza dal video vero al fotogramma {i:3d}: {distanza[i]:.5f}")
somma = sbagli.sum()
rapporto = distanza[299] / errore_vero
print(f"distanza finale / errore di un passo: {rapporto:.0f}")
print(f"somma degli sbagli di un passo fino al 299: {somma:.5f}")
print(f"distanza finale / somma degli sbagli: {distanza[299] / somma:.1f}")

### La finestra che scorre: la KV cache, la memoria e il primo fotogramma


In [ ]:
# KV cache di un DiT video: chiavi e valori di dimensione d, per strato, in bf16
modelli = {"Wan 1.3B": (1536, 30), "Wan 14B": (5120, 40)}   # (d, strati L)
byte = 2                                   # bf16
H, W, c_s, p = 480, 832, 8, 2              # 480p, VAE 8x spaziale, patch 2x2
S = (H // (c_s * p)) * (W // (c_s * p))    # token per fotogramma latente
GiB, GB = 2**30, 10**9                     # binario e decimale

print("token per fotogramma latente:", S)
for nome, (d, L) in modelli.items():
    per_token = 2 * d * byte * L           # K e V, in tutti gli strati
    per_fotogramma = S * per_token
    print(f"{nome}: {per_token} B per token, "
          f"{per_fotogramma / GiB:.2f} GiB per fotogramma latente")
    for n in (3, 6, 21):
        print(f"  {n:2d} fotogrammi latenti: "
              f"{n * per_fotogramma / GiB:6.2f} GiB = "
              f"{n * per_fotogramma / GB:6.2f} GB")

In [ ]:
T, w = 4, 6          # passi di denoising, finestra in fotogrammi latenti
fps, c_t = 16, 4     # fotogrammi al secondo e compressione temporale di Wan
S, d, L, byte = 1560, 5120, 40, 2          # Wan 14B a 480p
GiB = 2**30
per_fotogramma = S * 2 * d * byte * L      # byte di cache per fotogramma

# costo di ogni nuovo fotogramma, in coppie di fotogrammi che si guardano
print("per nuovo fotogramma, con T = 4 e w = 6:")
print("  finestra bidirezionale, T w^2    :", T * w**2)
print("  causale con ricalcolo, w^2 + T w :", w**2 + T * w)
print("  cache che scorre, T w            :", T * w)

print()
print("durata  latenti  cache cresce  cache finestra  coppie cresce"
      "  coppie scorre")
for secondi in (5, 30, 60, 300):
    Fl = 1 + fps * secondi // c_t             # F' = 1 + (F - 1) / c_t
    cresce = sum(T * i for i in range(Fl))     # i guarda i precedenti
    scorre = sum(T * min(i, w) for i in range(Fl))
    print(f"{secondi:>4} s {Fl:>8} {Fl * per_fotogramma / GiB:>10.1f} GiB"
          f" {w * per_fotogramma / GiB:>10.2f} GiB {cresce:>14,} {scorre:>13,}")

print(f"\nuna finestra di {w} fotogrammi latenti copre {w * c_t / fps:.1f} s "
      "di video")

## Il suono delle immagini

[Leggi la pagina](https://book.paithon.it/main/GenerazioneVideo/il-suono-delle-immagini.html)


### Due orologi e un orecchio esigente


In [ ]:
# Due orologi: frequenza dei token video (r_v) e audio (r_a), al secondo
soglia_ms = 25.0    # sfasamento che si puo' avvertire (MMAudio)
fotogramma_ms = 1000 / 24
print(f"un fotogramma a 24 fps dura {fotogramma_ms:.1f} ms; "
      f"la soglia e' {soglia_ms / fotogramma_ms:.2f} fotogrammi")

sistemi = {                           # nome: (r_v, r_a)
    "MMAudio": (8.0, 31.25),          # CLIP a 8 fps; latente audio a 31,25/s
    "Ovi": (24 / 4, 16000 / 512),     # VAE video 4x nel tempo; audio 16 kHz/512
    "LTX-2": (24 / 8, 25.0),          # 8x nel tempo: ipotesi, da LTX-Video
    "MiniMax H3": (24 / 4, 40.0),     # VAE video 4x; audio 40 token/s a canale
}
print(f"{'':11s}{'r_v':>6s}{'r_a':>7s}{'passo v':>10s}{'passo a':>9s}"
      f"{'25 ms in v':>12s}{'r_a/r_v':>9s}")
for nome, (rv, ra) in sistemi.items():
    print(f"{nome:11s}{rv:6.2f}{ra:7.2f}{1000 / rv:8.0f} ms{1000 / ra:6.0f} ms"
          f"{soglia_ms * rv / 1000:12.2f}{ra / rv:9.2f}")

# Ovi: 121 fotogrammi (5,04 s); il primo fotogramma latente sta da solo
rv, ra = sistemi["Ovi"]
lat_video = 1 + (121 - 1) // 4
tok_audio = -(-16000 * 5 // 512)      # 16 kHz x 5 s / 512 = 156,25, cioe' 157
s_conteggi = lat_video / tok_audio    # la scala che Ovi adotta
s_tassi = rv / ra                     # la scala dei soli tassi
print(f"Ovi: {lat_video} latenti video, {tok_audio} token audio "
      f"({tok_audio / lat_video:.2f} per latente)")
print(f"scala {lat_video}/{tok_audio} = {s_conteggi:.4f}, r_v/r_a = "
      f"{s_tassi:.4f}, scarto relativo {s_conteggi / s_tassi - 1:.1%}")
j = tok_audio - 1                     # indice dell'ultimo token audio
scarto_ms = j * (s_conteggi - s_tassi) / rv * 1000
print(f"token {j}: posizione video {j * s_conteggi:.2f} contro "
      f"{j * s_tassi:.2f}, cioe' {scarto_ms:.0f} ms")

# la stessa regola su clip piu' lunghi, con la scala ricalcolata su ciascuno
print(f"un latente video dura {1000 / rv:.0f} ms")
for durata in (5, 10, 20, 60):
    n_lat = 1 + (24 * durata) // 4    # 24 x durata + 1 fotogrammi
    n_tok = -(-16000 * durata // 512)
    scala = n_lat / n_tok
    scarto_ms = (n_tok - 1) * (scala - s_tassi) / rv * 1000
    print(f"{durata:3d} s: {n_lat:3d} latenti, {n_tok:4d} token, scarto "
          f"relativo {scala / s_tassi - 1:5.1%}, in fondo {scarto_ms:.0f} ms")

# MMAudio: le frequenze del flusso visivo scalate di 31,25/8
s = 31.25 / 8
print("MMAudio: scala", round(s, 3), "-> posizioni",
      [round(k * s, 2) for k in range(4)])

### La guida a due termini


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
d = 6
M = rng.normal(size=d)                   # uscita con testo e altra modalita'
M_senza_testo = rng.normal(size=d)       # testo tolto, altra modalita' presente
M_senza_altra = rng.normal(size=d)       # testo presente, altra modalita' tolta

def guida(s_T, s_M):
    """Guida a due termini: un'estrapolazione per ciascuna condizione tolta."""
    return M + s_T * (M - M_senza_testo) + s_M * (M - M_senza_altra)

for nome, (s_T, s_M) in {"video": (3, 3), "audio": (7, 3)}.items():
    esteso = (1 + s_T + s_M) * M - s_T * M_senza_testo - s_M * M_senza_altra
    print(f"{nome}: pesi {1 + s_T + s_M:+d} {-s_T:+d} {-s_M:+d}, somma "
          f"{(1 + s_T + s_M) - s_T - s_M}, le due scritture coincidono: "
          f"{np.allclose(guida(s_T, s_M), esteso)}")

# senza il secondo termine resta la guida di un solo termine, con w = 1 + s_T
w = 1 + 7
una_sola = M_senza_testo + w * (M - M_senza_testo)
print("con s_M = 0 resta la guida a un termine, w = 1 + s_T:",
      np.allclose(guida(7, 0), una_sola))

# le tre prove riassunte in un numero ciascuna (inventati), con s_T = 7, s_M = 3
completa, senza_testo, senza_altra = 10, 8, 9
spinte = completa + 7 * (completa - senza_testo) + 3 * (completa - senza_altra)
in_pesi = 11 * completa - 7 * senza_testo - 3 * senza_altra
print(f"tre numeri: {completa} + 7 x {completa - senza_testo} + 3 x "
      f"{completa - senza_altra} = {spinte}; in pesi: 11 x {completa} "
      f"- 7 x {senza_testo} - 3 x {senza_altra} = {in_pesi}")